# Figure 5c — Homotypic selectivity of PSI-IhN postsynaptic neurons

For every spinal cord neuron postsynaptic to a PSI-IhN **axon**, the *homotypic selectivity
index*: the fraction of that neuron's sensory input coming from the sensory subtype that
drives the presynaptic PSI-IhN. Low values mean feedforward inhibition is **heterotypic** —
the inhibited neuron listens to different afferents than the inhibitory neuron does.

One cumulative curve per PSI-IhN population. Curves shifted left = more heterotypic.

**Input.** `../data/psi_ihn_postsynaptic_feather/` (per-neuron SPINE tables) and
`../data/psi_ihn_partners_filtered_gt400.csv`. **Runs offline — no CAVE access.**

A neuron contacted by more than one PSI-IhN population is counted under **each** of them
(Methods), so the unit is the PSI-IhN → postsynaptic-neuron *connection*. The legend's
"numbers of PSI-IhN–postsynaptic neuron connections" are these counts.


In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "svg.fonttype": "none",
    "font.size": 6, "axes.titlesize": 6, "axes.labelsize": 6,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
})

CM = 1 / 2.54


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")
FEATHER_DIR = DATA / "psi_ihn_postsynaptic_feather"
PARTNERS_CSV = DATA / "psi_ihn_partners_filtered_gt400.csv"

# Only neurons postsynaptic to PSI-IhN *axons* (the axodendritic FFI motif).
KIND = "axon_pre"

SENSORY_TYPES = ["Aδ-HTMR", "C-Cold", "C-HTMR", "C-HTMR-Non-peptidergic",
                 "C-Heat", "Aβ-LTMR", "Aδ-LTMR", "C-LTMR"]
LOW_CONF_LABEL = "low_confidence_sensory"
CLASSIFIER_COLUMNS = ["cltmr", "abltmr", "adltmr", "chtmrnp",
                      "c-htmr", "adhtmrp", "cheat", "cold"]
LOW_CONF_THRESHOLD = 0.6

# Every sensory category that counts toward a neuron's denominator ("N/A" = non-sensory).
COUNTED_TYPES = SENSORY_TYPES + [LOW_CONF_LABEL]

TAGS = ["ad-htmr", "islet-chtmr", "islet-cltmr", "islet-adltmr", "IN-abltmr"]

TAG_LABELS = {
    "ad-htmr":      "PSI-IhN Aδ-HTMR → postsyn.",
    "islet-chtmr":  "Islet-PSI-IhN C-HTMR/Heat → postsyn.",
    "islet-cltmr":  "Islet-PSI-IhN C-LTMR → postsyn.",
    "islet-adltmr": "Islet-PSI-IhN Aδ-LTMR → postsyn.",
    "IN-abltmr":    "PSI-IhN Aβ-LTMR → postsyn.",
}

# The one sensory subtype each PSI-IhN population is itself driven by. The
# homotypic selectivity index is the fraction of a postsynaptic neuron's sensory
# input that comes from this subtype.
TAG_TARGET_TYPE = {
    "ad-htmr":      "Aδ-HTMR",
    "islet-chtmr":  "C-HTMR-Non-peptidergic",
    "islet-cltmr":  "C-LTMR",
    "islet-adltmr": "Aδ-LTMR",
    "IN-abltmr":    "Aβ-LTMR",
}

TAG_COLOURS = {
    "ad-htmr":      "#F28585",
    "islet-chtmr":  "#3C2F80",
    "islet-cltmr":  "#789342",
    "islet-adltmr": "#9AEBA3",
    "IN-abltmr":    "#45C4B0",
}

DISPLAY = {
    "C-Heat": "Peptidergic C-Nociceptors",
    "C-HTMR": "C-HTMR/Heat (SST)",
    "C-HTMR-Non-peptidergic": "C-HTMR/Heat (MrgprA3,MrgprD,MrgprB4)",
    LOW_CONF_LABEL: "Low-Confidence",
}

SENSORY_COLOURS = {
    "C-Cold":                               "#13678A",
    "Aδ-HTMR":                              "#F0AE44",
    "Peptidergic C-Nociceptors":            "#AE7F21",
    "C-HTMR/Heat (SST)":                    "#ffbbff",
    "C-HTMR/Heat (MrgprA3,MrgprD,MrgprB4)": "#548D95",
    "C-LTMR":                               "#789342",
    "Aδ-LTMR":                              "#9AEBA3",
    "Aβ-LTMR":                              "#45C4B0",
    "Low-Confidence":                       "#d3d3d3",
}


In [ ]:
def sensory_counts_for_neuron(post_root_id):
    """Sensory synapse counts for one postsynaptic neuron, or None if unusable.

    A synapse SPINE calls sensory but whose every subtype probability is
    <= LOW_CONF_THRESHOLD is relabelled `low_confidence_sensory` — it stays in the
    denominator as its own category rather than being dropped or forced elsewhere.
    """
    path = FEATHER_DIR / f"cell_type_predictions_{int(post_root_id)}.feather"
    if not path.exists():
        return None
    try:
        f = pd.read_feather(path)
    except Exception:
        return None
    if "cell_type" not in f.columns or "is_sensory" not in f.columns:
        return None
    if any(c not in f.columns for c in CLASSIFIER_COLUMNS):
        return None

    low_conf = (f[CLASSIFIER_COLUMNS] <= LOW_CONF_THRESHOLD).all(axis=1)
    f.loc[(f["is_sensory"] == True) & low_conf, "cell_type"] = LOW_CONF_LABEL

    vc = f["cell_type"].astype(str).value_counts().to_dict()
    counts = {ct: int(vc.get(ct, 0)) for ct in COUNTED_TYPES}
    if sum(counts.values()) <= 0:
        return None          # readable but nothing classifiable — skip, don't keep as zeros
    return counts


def build_postsynaptic_table():
    """One row per (PSI-IhN population, postsynaptic neuron) with its sensory composition.

    A postsynaptic neuron contacted by more than one population is assigned to the
    population supplying the most synapses, so each neuron appears once per figure.
    """
    partners = pd.read_csv(PARTNERS_CSV)
    partners = partners[partners["kind"] == KIND]
    partners = partners[partners["tag"].isin(TAGS)]

    # Collapse repeated (tag, post) edges. A neuron contacted by more than one PSI-IhN
    # population stays in EACH of them -- per the Methods, "a single spinal cord neuron
    # could belong to multiple postsynaptic neuron groups". So the unit here is the
    # PSI-IhN -> postsynaptic-neuron *connection*, not the neuron.
    edges = (partners.groupby(["tag", "post_root_id"], as_index=False)["n_synapses"]
                     .sum())

    rows, skipped = [], 0
    for _, e in edges.iterrows():
        counts = sensory_counts_for_neuron(e.post_root_id)
        if counts is None:
            skipped += 1
            continue
        denom = sum(counts.values())
        row = {"tag": e.tag, "post_root_id": int(e.post_root_id),
               "n_synapses_from_tag": int(e.n_synapses), "n_sensory": denom}
        row.update(counts)
        row["target_frac"] = counts[TAG_TARGET_TYPE[e.tag]] / denom
        rows.append(row)

    table = pd.DataFrame(rows)
    print(f"{len(edges)} PSI-IhN -> postsynaptic connections "
          f"({edges.post_root_id.nunique()} distinct neurons); "
          f"{len(table)} usable, {skipped} skipped "
          f"(feather missing or no classifiable sensory synapses)")
    print(table.tag.value_counts().reindex(TAGS).to_string())
    return table


postsyn = build_postsynaptic_table()


In [ ]:
import matplotlib.pyplot as plt


def ecdf(values):
    x = np.sort(np.asarray(values, float))
    return x, np.arange(1, len(x) + 1) / len(x)


fig, ax = plt.subplots(figsize=(8 * CM, 4 * CM))

for tag in TAGS:
    vals = postsyn.loc[postsyn.tag == tag, "target_frac"].dropna().to_numpy()
    if len(vals) == 0:
        continue
    x, y = ecdf(vals)
    ax.step(x, y, where="post", linewidth=0.9, color=TAG_COLOURS[tag],
            label=f"{TAG_LABELS[tag]} (n={len(vals)})")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Selectivity")
ax.set_ylabel("Cumulative proportion")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.tick_params(axis="both", width=0.4, length=2)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=4, frameon=False,
          handlelength=1.6, labelspacing=0.4, borderaxespad=0.0)
fig.subplots_adjust(right=0.72)
fig.tight_layout(pad=0.4)
fig.savefig("figure5c_homotypic_selectivity_ecdf.svg", format="svg")
plt.show()


In [ ]:
# Median selectivity per population, and the fraction of postsynaptic neurons that are
# themselves homotypic (>50% of their sensory input from the PSI-IhN's own subtype).
print(f"{'population':42s}{'n':>5s}{'median':>9s}{'>50% homotypic':>16s}")
for tag in TAGS:
    v = postsyn.loc[postsyn.tag == tag, "target_frac"].dropna()
    print(f"{TAG_LABELS[tag]:42s}{len(v):5d}{v.median():9.3f}{(v > 0.5).mean():15.1%}")
